# 缺少故障标签时怎样发现异常传感器记录？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

{'zscore': {'alerts': 29, 'precision': 0.5862068965517241, 'recall': 0.85}, 'isolation_forest': {'alerts': 29, 'precision': 0.6206896551724138, 'recall': 0.9}, 'threshold': 0.5565205678108183}


## 1. 补全：复核队列排序
找到分数最高的前 5 条，检查分数为非递增。

In [2]:
top = result.nlargest(5,'score')
assert len(top)==5 and np.all(np.diff(top['score'])<=0)
print(top)

     temperature  vibration    current     score  alert  simulated_truth
125    50.129203   3.152961  13.368429  0.769007   True                1
117    51.531194   3.924595  12.839757  0.765891   True                1
49     46.482535   3.935408  12.988662  0.744466   True                1
140    47.896778   3.077439  12.596463  0.737267   True                1
136    44.890664   3.016726  13.280451  0.735026   True                1


## 2. 修改：复核容量减半
将历史阈值设为 97.5% 分位数，验证报警数量不多于原方案。

In [3]:
strict = test_scores > np.quantile(train_scores,0.975)
assert strict.sum() <= alert.sum()
print(score_alert(strict))

{'alerts': 23, 'precision': 0.7391304347826086, 'recall': 0.85}


## 3. 迁移：整体温度升高不等于设备故障
把测试温度整体提高 8°C，保持其他特征不变；比较报警数，说明环境变化需要复核分布。

In [4]:
shifted = test.copy(); shifted[:,0] += 8
shifted_alert = -model.score_samples(shifted) > threshold
assert shifted_alert.shape == alert.shape
print({'原报警数':int(alert.sum()),'升温后报警数':int(shifted_alert.sum())})

{'原报警数': 29, '升温后报警数': 140}
